# R1a — SFT complet, un bras

Première étape d'un bras : le *supervised fine-tuning* sur Aya haoussa. Le DPO suit dans une
soumission séparée (`R1b`), parce qu'un bras entier coûte 9,76 h contre un plafond de
session Kaggle de 9 h.

## Paramètres du run

Les deux seules cellules à modifier d'un bras à l'autre sont juste en dessous. **Le reste
est identique entre A2 et A3** — c'est ce qui rend l'écart attribuable au backbone et à rien
d'autre.

| | |
| :---- | :---- |
| Durée attendue | ~5,5 h (352 pas à 56,8 s, mesuré) |
| VRAM attendue | ~7,5 Go sur 15,64 |
| Sortie | adaptateur LoRA dans `/kaggle/working/results/sft_checkpoints/final` |

## Résilience

Le run sauvegarde un checkpoint tous les 20 pas et écrit ses métriques à chaque log. Une
session coupée perd au pire ~19 minutes, et relancer ce notebook **reprend automatiquement**
au dernier checkpoint.

### Réglages Kaggle
Accelerator **T4 x2**, internet activé, datasets `afrique-safety-dpo-code` et
`afrique-safety-dpo-data` attachés.

## 0. Ce qui change d'un bras à l'autre

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
BRAS   = "A3"                                        # A3 = cible, A2 = controle
MODELE = "McGill-NLP/AfriqueQwen3.5-4B-50Langs"      # A2: "Qwen/Qwen3.5-4B-Base"
GRAINE = 42
# ─────────────────────────────────────────────────────────────────────────────
print(f"bras {BRAS} | {MODELE} | graine {GRAINE}")

In [ ]:
!pip install -q -U "transformers==5.16.1" "trl==1.12.0" "peft==0.20.0" bitsandbytes accelerate datasets

In [ ]:
import os, sys
from pathlib import Path

# Un seul GPU visible: Trainer active DataParallel des qu'il en voit plusieurs, ce qui
# doublerait silencieusement le nombre de sequences par pas.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

CODE_DS = "/kaggle/input/afrique-safety-dpo-code"
sys.path.insert(0, CODE_DS if Path(CODE_DS).exists() else str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "paths.py").exists())
))

from src.paths import resolve_roots

R = resolve_roots()
ROOT, SORTIE = R["code"], R["output"]
print("code    :", ROOT)
print("sorties :", SORTIE)

import torch, transformers, trl, peft
print(f"\n{torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} Go")
print(f"transformers {transformers.__version__} | trl {trl.__version__} | peft {peft.__version__}")

## 1. Données — Aya haoussa

Nativement rédigé par des locuteurs, Apache-2.0. Le découpage est celui déjà testé, avec la
même graine : ce qui part en SFT ne doit jamais reapparaitre dans l'évaluation ni dans le DPO.

In [ ]:
import yaml
from datasets import load_dataset

from src.data import build_aya_sft_examples, split_by_base_stem

config = yaml.safe_load(open(ROOT / "config.yaml"))
config["training"]["seed"] = GRAINE
config["model"]["base_model_name"] = MODELE
config["paths"]["output_dir"] = str(SORTIE / "results") + "/"

LANGUE = config["sft"]["language"]
aya = load_dataset("CohereLabs/aya_dataset", split="train")
aya_ha = aya.filter(lambda r: r["language"] == LANGUE)   # filtrer avant de materialiser
exemples = build_aya_sft_examples(list(aya_ha), LANGUE)

# GRAINE_SPLIT est volontairement FIXE, independante de GRAINE. La graine
# d'entrainement fait varier l'initialisation LoRA et l'ordre des exemples; si elle
# faisait aussi varier la partition, les trois graines melangeraient variance
# d'entrainement et variance de partition, et l'ecart A3-A2 ne serait plus attribuable.
GRAINE_SPLIT = 42
tr_sft, ev_sft = split_by_base_stem(exemples, seed=GRAINE_SPLIT)
print(f"{LANGUE} : {len(exemples)} exemples -> {len(tr_sft)} train / {len(ev_sft)} eval")
assert not ({p["base_stem"] for p in tr_sft} & {p["base_stem"] for p in ev_sft})
print("contamination train <-> eval : 0")

In [ ]:
# La liste d'evaluation est sauvegardee AVANT l'entrainement: sans elle, l'adaptateur
# produit serait inevaluable sur la meme partition, et refaire le split plus tard avec une
# autre graine reintroduirait la contamination qu'on vient de verifier.
import json

(SORTIE / "results").mkdir(parents=True, exist_ok=True)
(SORTIE / "results" / f"sft_eval_stems_split{GRAINE_SPLIT}.json").write_text(
    json.dumps(sorted({p["base_stem"] for p in ev_sft}), ensure_ascii=False), encoding="utf-8"
)
print(f"partition d'evaluation sauvegardee (graine de split {GRAINE_SPLIT}, fixe)")

## 2. Entraînement

`run_sft` place un checkpoint tous les 20 pas et écrit ses métriques à chaque log. Si cette
cellule est interrompue, relancer le notebook reprend au dernier checkpoint plutôt que de
tout recommencer.

In [ ]:
import time

import torch

from src.train import run_sft

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
depart = time.time()
trainer = run_sft(config, tr_sft)          # pas de max_steps: run complet
duree = time.time() - depart
vram = torch.cuda.max_memory_allocated() / 1e9

print(f"\ndurée : {duree/3600:.2f} h")
print(f"VRAM  : {vram:.2f} Go")

## 3. Ce que le run laisse derrière lui

In [ ]:
resume = {
    "bras": BRAS,
    "modele": MODELE,
    "graine": GRAINE,
    "exemples_sft": len(tr_sft),
    "duree_h": round(duree / 3600, 3),
    "vram_crete_go": round(vram, 2),
    "pas": trainer.state.global_step,
    "loss_finale": next(
        (e["loss"] for e in reversed(trainer.state.log_history) if "loss" in e), None
    ),
    "adaptateur": str(SORTIE / "results" / "sft_checkpoints" / "final"),
}
(SORTIE / "results" / f"sft_resume_{BRAS}_s{GRAINE}.json").write_text(
    json.dumps(resume, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(json.dumps(resume, indent=2, ensure_ascii=False))

In [ ]:
# La courbe de loss: on regarde qu'elle descende, pas ou elle atterrit. Une loss plate
# signalerait un adaptateur attache a rien -- le mode d'echec silencieux du QLoRA.
import matplotlib.pyplot as plt

points = [(e["step"], e["loss"]) for e in trainer.state.log_history if "loss" in e]
if points:
    pas, pertes = zip(*points)
    plt.figure(figsize=(7, 3))
    plt.plot(pas, pertes)
    plt.xlabel("pas"); plt.ylabel("loss"); plt.title(f"SFT {BRAS} — graine {GRAINE}")
    plt.grid(alpha=.3); plt.show()
    print(f"loss : {pertes[0]:.3f} -> {pertes[-1]:.3f}")
    if pertes[-1] >= pertes[0]:
        print("ATTENTION: la loss n'a pas baisse. Verifier que l'adaptateur est bien attache.")

In [ ]:
!ls -la {SORTIE}/results/sft_checkpoints/final/ 2>/dev/null | head -8

---

## Suite

**R1b** reprend l'adaptateur produit ici et lance le DPO. Il faut lui renseigner, dans
`config.yaml` ou dans le notebook, `dpo.adapter_path` pointant vers
`sft_checkpoints/final`.

⚠️ Le DPO doit charger cet adaptateur avec `is_trainable=True` et **sans** `peft_config` :
sans quoi il s'entraînerait à vide ou empilerait un second adaptateur sur le premier resté
gelé. Les deux cas tournent sans erreur et produisent un résultat faux — c'est traité dans
`load_causal_lm`, et verrouillé par les tests de `test_handoff.py`.

Pour lancer le bras de contrôle : changer `BRAS` et `MODELE` dans la première cellule. Tout
le reste, graine comprise, doit rester identique.